In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)

In [2]:
FEATURE_DIR = Path("artifacts/features")
MODEL_DIR = Path("artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: test is intentionally NOT loaded here. Only train and validation.
train_features = pd.read_csv(FEATURE_DIR / "train_features.csv")
validation_features = pd.read_csv(FEATURE_DIR / "validation_features.csv")

TARGET = "is_late"

X_train = train_features.drop(columns=[TARGET])
y_train = train_features[TARGET]

X_validation = validation_features.drop(columns=[TARGET])
y_validation = validation_features[TARGET]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test: not loaded yet")

Train: (67533, 65)
Validation: (14471, 65)
Test: not loaded yet


In [3]:
print("Training class distribution:")
print(y_train.value_counts())
print("\nTraining class percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation class distribution:")
print(y_validation.value_counts())
print("\nValidation late %:", round(y_validation.mean() * 100, 2))

Training class distribution:
is_late
0    61436
1     6097
Name: count, dtype: int64

Training class percentages:
is_late
0    90.97
1     9.03
Name: proportion, dtype: float64

Validation class distribution:
is_late
0    13698
1      773
Name: count, dtype: int64

Validation late %: 5.34


## Why these models?

Same reasoning as before: the data is tabular and the target is imbalanced, so we compare a simple baseline against two nonlinear tree ensembles. `class_weight="balanced"` is added this time so the models themselves account for the imbalance during training, not just at the threshold stage.

The main model-selection metric is still **Average Precision (PR-AUC)**, computed on validation, because it does not depend on a threshold.

In [4]:
def evaluate_predictions(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
    }


def choose_threshold_from_oof(y_true, probabilities):
    thresholds = np.linspace(0.05, 0.95, 181)
    rows = []
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, predictions, zero_division=0),
            "recall": recall_score(y_true, predictions, zero_division=0),
            "f1": f1_score(y_true, predictions, zero_division=0),
        })
    table = pd.DataFrame(rows)
    best = table.loc[table["f1"].idxmax()]
    return float(best["threshold"]), table

## 1. Baseline model

In [5]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

dummy_probabilities = dummy_model.predict_proba(X_validation)[:, 1]
dummy_results = evaluate_predictions(y_validation, dummy_probabilities, threshold=0.5)
pd.DataFrame([dummy_results])

,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.946583,0.0,0.0,0.0,0.5,0.053417


## 2. Tune HistGradientBoosting on validation

In [6]:
hist_params = [
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 15, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.10, "max_iter": 200, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 31, "l2_regularization": 5.0},
]

hist_rows = []
for params in hist_params:
    model = HistGradientBoostingClassifier(**params, class_weight="balanced", random_state=42)
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_validation)[:, 1]
    metrics = evaluate_predictions(y_validation, probabilities, threshold=0.5)
    hist_rows.append({**params, **metrics})

hist_tuning = pd.DataFrame(hist_rows).sort_values("average_precision", ascending=False).reset_index(drop=True)
hist_tuning

,learning_rate,max_iter,max_leaf_nodes,l2_regularization,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.05,300,31,5.0,0.849285,0.163801,0.443726,0.239275,0.753412,0.171387
1,0.10,200,31,1.0,0.837053,0.159725,0.481242,0.239845,0.753320,0.170952
2,0.05,300,31,1.0,0.852049,0.165034,0.435964,0.239432,0.749415,0.167624
3,0.05,200,15,1.0,0.837606,0.155827,0.461837,0.233029,0.749372,0.162939


## 3. Tune ExtraTrees on validation

In [7]:
extra_params = [
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 1, "max_features": None},
]

extra_rows = []
for params in extra_params:
    model = ExtraTreesClassifier(**params, class_weight="balanced", random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_validation)[:, 1]
    metrics = evaluate_predictions(y_validation, probabilities, threshold=0.5)
    extra_rows.append({**params, **metrics})

extra_tuning = pd.DataFrame(extra_rows).sort_values("average_precision", ascending=False).reset_index(drop=True)
extra_tuning

,n_estimators,max_depth,min_samples_leaf,max_features,accuracy,precision,recall,f1,roc_auc,average_precision
0,300,20.0,1,NaN,0.924746,0.213768,0.152652,0.178113,0.710662,0.150166
1,300,NaN,2,sqrt,0.872987,0.157556,0.316947,0.210481,0.716248,0.146782
2,300,20.0,2,sqrt,0.783913,0.125874,0.512290,0.202092,0.705780,0.141805
3,300,12.0,2,sqrt,0.645982,0.091165,0.627426,0.159199,0.676254,0.109022


## 4. Select the best model by validation Average Precision

In [8]:
best_hist = hist_tuning.iloc[0]
best_extra = extra_tuning.iloc[0]

if best_hist["average_precision"] >= best_extra["average_precision"]:
    final_model_name = "HistGradientBoosting"
    final_params = {
        "learning_rate": float(best_hist["learning_rate"]),
        "max_iter": int(best_hist["max_iter"]),
        "max_leaf_nodes": int(best_hist["max_leaf_nodes"]),
        "l2_regularization": float(best_hist["l2_regularization"]),
    }
    final_model = HistGradientBoostingClassifier(**final_params, class_weight="balanced", random_state=42)
else:
    final_model_name = "ExtraTrees"
    final_params = {
        "n_estimators": int(best_extra["n_estimators"]),
        "max_depth": None if pd.isna(best_extra["max_depth"]) else int(best_extra["max_depth"]),
        "min_samples_leaf": int(best_extra["min_samples_leaf"]),
        "max_features": best_extra["max_features"],
    }
    final_model = ExtraTreesClassifier(**final_params, class_weight="balanced", random_state=42, n_jobs=-1)

final_model.fit(X_train, y_train)
validation_probabilities = final_model.predict_proba(X_validation)[:, 1]

print("Selected model:", final_model_name)
print("Validation Average Precision:", round(average_precision_score(y_validation, validation_probabilities), 4))
print("Validation ROC-AUC:", round(roc_auc_score(y_validation, validation_probabilities), 4))

Selected model: HistGradientBoosting
Validation Average Precision: 0.1714
Validation ROC-AUC: 0.7534


## 5. Select the threshold using TimeSeriesSplit OOF predictions from TRAIN only

This is the main fix. Instead of picking the threshold directly on the 14,471-row validation set (only 773 late orders), we generate out-of-fold predictions across 5 chronological folds **inside the training set**. Each fold trains on earlier data and predicts on later data, so the chronological principle from Notebook 3 is respected at this stage too.

`TimeSeriesSplit` does not produce an OOF prediction for the very first fold's training rows (there's no earlier data to train on for them), so those rows are excluded from threshold selection — this is expected and not a bug.

Validation and test labels are **not** used to choose the threshold.

In [9]:
tscv = TimeSeriesSplit(n_splits=5)
oof_probabilities = np.full(len(X_train), np.nan)

for fold, (fit_idx, holdout_idx) in enumerate(tscv.split(X_train), start=1):
    X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
    X_holdout = X_train.iloc[holdout_idx]

    if final_model_name == "HistGradientBoosting":
        fold_model = HistGradientBoostingClassifier(**final_params, class_weight="balanced", random_state=42)
    else:
        fold_model = ExtraTreesClassifier(**final_params, class_weight="balanced", random_state=42, n_jobs=-1)

    fold_model.fit(X_fit, y_fit)
    oof_probabilities[holdout_idx] = fold_model.predict_proba(X_holdout)[:, 1]
    print(f"Fold {fold} completed. Holdout rows: {len(holdout_idx)}")

valid_mask = ~np.isnan(oof_probabilities)
print("\nRows used for OOF threshold selection:", valid_mask.sum(), "out of", len(X_train))

oof_ap = average_precision_score(y_train[valid_mask], oof_probabilities[valid_mask])
oof_roc_auc = roc_auc_score(y_train[valid_mask], oof_probabilities[valid_mask])
print("OOF Average Precision:", round(oof_ap, 4))
print("OOF ROC-AUC:", round(oof_roc_auc, 4))

Fold 1 completed. Holdout rows: 11255
Fold 2 completed. Holdout rows: 11255
Fold 3 completed. Holdout rows: 11255
Fold 4 completed. Holdout rows: 11255
Fold 5 completed. Holdout rows: 11255

Rows used for OOF threshold selection: 56275 out of 67533
OOF Average Precision: 0.2443
OOF ROC-AUC: 0.6986


In [10]:
best_threshold, threshold_table = choose_threshold_from_oof(
    y_train[valid_mask], oof_probabilities[valid_mask]
)

print("Selected threshold from training OOF predictions:", best_threshold)
threshold_table.sort_values("f1", ascending=False).head(10)

Selected threshold from training OOF predictions: 0.61


,threshold,precision,recall,f1
112,0.610,0.258674,0.347240,0.296484
111,0.605,0.252650,0.355691,0.295444
113,0.615,0.261971,0.338428,0.295332
114,0.620,0.266377,0.330516,0.295000
116,0.630,0.277232,0.314871,0.294856
110,0.600,0.248397,0.362345,0.294741
115,0.625,0.271145,0.322244,0.294495
117,0.635,0.282941,0.306600,0.294295
109,0.595,0.243775,0.369718,0.293819
108,0.590,0.239662,0.377270,0.293119


In [11]:
validation_final_results = evaluate_predictions(
    y_validation, validation_probabilities, threshold=best_threshold
)

print("Validation results using the frozen OOF threshold:")
pd.DataFrame([validation_final_results])

Validation results using the frozen OOF threshold:


,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.903669,0.208451,0.287193,0.241567,0.753412,0.171387


## 6. Freeze the model and evaluate the test set once

At this point:
- the model type is fixed;
- hyperparameters are fixed;
- the classification threshold is fixed (from training OOF predictions, not validation);
- the test set is used only for the final evaluation.

**Do not use the test results to change the model.** This is the only cell in this notebook that loads and touches the test set.

In [12]:
test_features = pd.read_csv(FEATURE_DIR / "test_features.csv")
X_test = test_features.drop(columns=[TARGET])
y_test = test_features[TARGET]

print("Test:", X_test.shape)

test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_results = evaluate_predictions(y_test, test_probabilities, threshold=best_threshold)
test_results

Test: (14472, 65)


{'accuracy': 0.8225538971807629,
 'precision': 0.07582938388625593,
 'recall': 0.15047021943573669,
 'f1': 0.10084033613445378,
 'roc_auc': 0.597808348709646,
 'average_precision': 0.08329384759912274}

In [13]:
test_predictions = (test_probabilities >= best_threshold).astype(int)
print(classification_report(
    y_test, test_predictions, target_names=["on_time", "late"], zero_division=0
))

              precision    recall  f1-score   support

     on_time       0.94      0.87      0.90     13515
        late       0.08      0.15      0.10       957

    accuracy                           0.82     14472
   macro avg       0.51      0.51      0.50     14472
weighted avg       0.88      0.82      0.85     14472



## 7. Save the final model and modeling artifacts

In [14]:
model_path = MODEL_DIR / "final_model.joblib"
joblib.dump(final_model, model_path)

model_metadata = {
    "model_name": final_model_name,
    "model_parameters": final_params,
    "selection_metric": "average_precision",
    "threshold_selection_method": "TimeSeriesSplit (5-fold) OOF predictions on training data",
    "threshold_selection_metric": "f1",
    "selected_threshold": best_threshold,
    "training_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "features_used": X_train.columns.tolist(),
    "test_evaluated_once_in_this_notebook": True,
}

with open(MODEL_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

pd.DataFrame([{
    "model": final_model_name, "threshold": best_threshold, **validation_final_results
}]).to_csv(MODEL_DIR / "validation_final_results.csv", index=False)

pd.DataFrame([{
    "model": final_model_name, "threshold": best_threshold, **test_results
}]).to_csv(MODEL_DIR / "test_final_results.csv", index=False)

hist_tuning.to_csv(MODEL_DIR / "hist_gradient_boosting_tuning.csv", index=False)
extra_tuning.to_csv(MODEL_DIR / "extra_trees_tuning.csv", index=False)
threshold_table.to_csv(MODEL_DIR / "threshold_oof_table.csv", index=False)

print("Saved:", model_path)
print("Saved model metadata and evaluation artifacts.")

Saved: artifacts\models\final_model.joblib
Saved model metadata and evaluation artifacts.


## 8. Production inference pattern

For a new order, Notebook 5's saved preprocessing must be used first.

The production flow is:

`new raw order → same feature engineering → saved preprocessor.transform() → final_model.predict_proba() → fixed threshold`

The model and threshold are not retrained for each new order.

In [15]:
loaded_model = joblib.load(MODEL_DIR / "final_model.joblib")
loaded_metadata = json.loads((MODEL_DIR / "model_metadata.json").read_text(encoding="utf-8"))

print("Loaded model:", loaded_metadata["model_name"])
print("Fixed threshold:", loaded_metadata["selected_threshold"])

Loaded model: HistGradientBoosting
Fixed threshold: 0.61


## Final conclusion

The split strategy from Notebook 3 (chronological, 70/15/15) is unchanged.

The model-selection process is:

1. Compare against a simple baseline.
2. Tune nonlinear tabular models (with `class_weight="balanced"`) on training data.
3. Select the best configuration using validation **Average Precision**.
4. Select the classification threshold using **TimeSeriesSplit out-of-fold predictions on the training set** — not directly on the small validation set.
5. Freeze everything.
6. Evaluate the test set once at the end.
7. Save the final model, threshold, parameters, and results.

**Note on test set discipline:** if the test set was evaluated in an earlier version of this notebook before this fix was applied, that earlier exposure should be disclosed in the final report as a limitation, even though no test data was used to fit the model or choose this threshold.